# Week 11 Problem Set: The Endorsement

You are the political director from the case. Before the board meets you (1) check whether the analyst's probabilities can be trusted, (2) build the decision, and (3) write your recommendation.

**Lying with data, the checklist so far:**
1. **W1:** Conflating fixed and marginal costs.
2. **W2:** Presenting an observational comparison as a causal effect.
3. **W3:** Applying a result from one setting to a different one.
4. **W4:** Cherry-picking the winning arm from a multi-arm test.
5. **W5:** Treating an underpowered null as evidence of no effect.
6. **W7:** Reporting the complier comparison as a causal effect.
7. **W8:** Cherry-picking polls; house effects; ignoring nonresponse bias.
8. **W9:** Comparing by effect size without cost; ignoring uncertainty in cost per vote.
9. **W10:** An accuracy number quoted with no floor beside it, a model graded on the data it was built from, and a model of **who votes** sold as a model of **who you can move**.
10. **W11:** Quoting a probability with no calibration record behind it, and quoting a **conditional** probability without the number it is conditional on.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

In [ ]:
import pandas as pd
import numpy as np

rec = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                 'main/weeks/wk11_forecasts_and_calibration/data/analyst_record.csv')
print(rec.shape)
rec.head()

## Task 1: Is the analyst calibrated? (you write the code)

The bins are pre-filled. **You** build the table, using the same `pd.cut` + `groupby` from livecode.

In [ ]:
rec['bin'] = pd.cut(rec['forecast'], bins=np.arange(0, 1.01, 0.2))

# YOUR CODE HERE: group `rec` by 'bin' and, for each bin, compute the mean forecast,
# the actual fraction that happened, and the number of races. Use observed=True.
# Assign the result to `calibration`.
calibration =  # YOUR CODE HERE

print(calibration.round(3))

*Check: five bins holding 27, 36, 26, 41 and 20 races. In the 0.6-0.8 bin she says about 0.72 and it happens about 0.61.*

**Question 1.** Is she calibrated? Answer in two sentences, and cite one bin where she is and one where she is not.

*Your answer:*

## Task 2: Does she beat guessing? (pre-filled)

Run the cell.

In [ ]:
brier_analyst  = np.mean((rec['forecast'] - rec['won'])**2)
base_rate      = rec['won'].mean()
brier_baseline = np.mean((base_rate - rec['won'])**2)

print('base rate (how often anyone wins at all):', round(base_rate, 3))
print('Brier, her forecasts: ', round(brier_analyst, 4))
print('Brier, base rate only:', round(brier_baseline, 4))

**Question 2.** She is overconfident at the high end and she still beats the base rate. Explain in two or three sentences how both can be true at once, and what that means for whether the board should keep paying her.

**Before you move on:** the case's forecast is 60/40. Look back at your Task 1 table and say whether that is a part of her record you would trust. One sentence.

*Careful with the bin labels.* `pd.cut` closes each bin on the right, so a forecast of exactly 0.60 falls in the bin printed as `(0.4, 0.6]` — **not** the one printed as `(0.6, 0.8]`. Run the cell below if you want the two halves separated cleanly.

In [ ]:
for name, part in [('60% and below', rec[rec['forecast'] <= 0.60]),
                   ('above 60%   ', rec[rec['forecast'] >  0.60])]:
    print(f'{name}: {len(part)} races, she said {part.forecast.mean():.3f}, '
          f'happened {part.won.mean():.3f}')

*Your answers:*

## Task 3: Build the decision (you write the code)

The four inputs and the expected values are pre-filled. **You** write the break-even.

In [ ]:
p_A, gen_A, signs_A = 0.60, 0.45, 0.40
p_B, gen_B, signs_B = 0.40, 0.55, 1.00

neutral = p_A * gen_A * signs_A + p_B * gen_B * signs_B

def endorse_B(d, signs_A_annoyed=0.30):
    a = (p_A - d) * gen_A * signs_A_annoyed
    b = (p_B + d) * gen_B * signs_B
    return a + b

print('stay neutral:', round(neutral * 100, 2), '%')
print('endorse B, move nothing:', round(endorse_B(0) * 100, 2), '%')

In [ ]:
# YOUR CODE HERE: how far behind you start, and what one point buys.
# Then divide the first by the second to get the break-even, in points.
behind    =  # YOUR CODE HERE
per_point =  # YOUR CODE HERE

print('behind by:  ', round(behind * 100, 2), 'points of ordinance')
print('each point buys:', round(per_point * 100, 3))
print('break-even: ', round(behind / per_point, 1), 'points of primary chance')

*Check: 2.7 behind, 0.415 a point, break-even 6.5.*

**Before you move on:** the break-even is in points of B's **chance of winning the primary**. Why is that not the same as moving his vote share by 6.5 points? One or two sentences.

*Your answer:*

## Task 4: What is the answer resting on? (pre-filled)

We invented the 30\% that A signs once you cross her. Sweep it.

In [ ]:
# This sweeps how much A signs ONCE CROSSED, holding her uncrossed 40% fixed.
# (The slide's other row moves her uncrossed share instead, and the crossed value
#  scales with it -- a quarter less. Different knob, different answer.)
for s in [0.40, 0.35, 0.30, 0.25, 0.20]:
    behind_s = neutral - endorse_B(0, signs_A_annoyed=s)
    per_pt_s = endorse_B(0.01, signs_A_annoyed=s) - endorse_B(0, signs_A_annoyed=s)
    print(f'A signs {s:.2f} once annoyed -> break-even {behind_s / per_pt_s:5.1f} points')

*Check: 0.0, 3.4, 6.5, 9.3, 11.7.*

**Question 3.** At 0.40 the break-even is exactly zero. Say in two sentences what that means, and why it makes this the number you would most want to pin down before the board votes.

*Your answer:*

## Task 5: The memo (250-350 words, plus a research request)

Write to your board. This memo has two parts and they are graded as one piece of work.

**Part 1, the recommendation (250-350 words).** Endorse A, endorse B, or stay neutral. Lead with the recommendation, then the reasons. You must use all three of these:

- **(a) The arithmetic.** What each candidate is worth to you in expectation, and where those numbers come from. Use at least two specific numbers.
- **(b) The analyst.** What her track record showed, and what you now believe about the 60/40 because of it. Do not skip this because it came out in her favour.
- **(c) The steelman.** The best version of the case against your recommendation, stated fairly, and why you are not persuaded.

**Part 2, the research request (about 100 words, labelled).** Name the **one number** you would spend the next two weeks pinning down. Say (1) how you would actually go and measure it, and (2) what value it would have to take for you to change your recommendation. Be specific: a number, not "more research."

**Style rules:**
- Recommendation in the first sentence.
- Every number you use should be one a board member could check against something you showed them.
- The research request must name a value that would flip you. "It depends" is not an answer.

**Memo to:** Board of Directors
**From:** Political Director
**Re:** Mayoral primary endorsement

*Replace this text with your memo, then your labelled research request.*

---

**Due at 4:00pm on Wednesday Dec 2**, to the **problem set** assignment on Canvas. Whatever you had at 5:55pm in class already went to the separate **in-class** assignment; that one is your attendance credit and you do not resubmit it.


## Before you submit

1. **Runtime → Restart session and run all.** Do this *after* you have finished every task and written your memo. It clears every variable and runs the notebook from top to bottom, in order, so the version you hand in is one that actually works start to finish.
2. **Check that every cell actually ran.** Scroll from the top. Every code cell should show a number in its left margin and its output below it. If the run stopped at a cell with an error, that is a cell you have not finished. Fix it, then restart and run all again.
3. **File → Print → Save as PDF.**
4. **Open the PDF and read it before you upload.** The PDF will look complete even when it isn't. Every heading and prompt prints whether or not the code ran. What matters is the **output**: under each code cell you should see a table or a number. A red error box, or `In [ ]` with nothing beneath it, means that part did not run and will be graded as missing. Also check that your memo printed in full.
5. Upload the PDF to Canvas.